[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/AI%E5%8D%94%E4%BD%9C%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92_%E8%81%B7%E5%A0%B4%E5%AF%A6%E6%88%B0%E7%AF%87/02_%E9%9B%BB%E5%95%86_%E5%95%86%E5%93%81%E9%80%80%E8%B2%A8%E9%A2%A8%E9%9A%AA%E9%A0%90%E6%B8%AC/%E5%95%86%E5%93%81%E9%80%80%E8%B2%A8%E9%A2%A8%E9%9A%AA%E9%A0%90%E6%B8%AC_%E5%88%86%E9%A1%9E%E7%AF%87.ipynb)

# 🛒 範例 02：電商營運—商品退貨風險預測與逆物流防範 (分類篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「電商營運、倉儲物流主管與 1 人電商賣家（無程式背景）」**設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以電商實務白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說商業決策邏輯！

---

## 📊 一、電商資料集介紹 (`returns.csv`)

本數據集記錄了電商平台歷年 2,000 筆訂單明細與消費者最終退貨紀錄：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 電商業務實務意涵 |
| :--- | :--- | :--- | :--- |
| **商品類別** | 類別型 (`object`) | 流行服飾、鞋包配件、3C數位家電、居家生活、美妝保養 | 服飾鞋包退貨率通常最高 |
| **單價_TWD** | 數值型 (`int64`) | 1,980 (元) | 高單價商品消費者審視標準較高 |
| **折扣幅度** | 數值型 (`float64`) | 0.3 (即 7 折促銷) | 大促銷促成衝動購物 |
| **是否首購顧客** | 類別型 (`object`) | 是、否 | 新客對版型不熟悉易有尺寸問題 |
| **過往退貨率** | 數值型 (`float64`) | 0.45 | 該顧客歷史退貨傾向指標 |
| **購買數量** | 數值型 (`int64`) | 2 (件) | 同款多件常為包色試穿徵兆 |
| **付款方式** | 類別型 (`object`) | 信用卡、超商取貨付款、LINE Pay、貨到付款 | 付款方式反映付款承諾度 |
| **配送方式** | 類別型 (`object`) | 黑貓宅急便、超商純取貨、品牌自配送 | 物流履約管道 |
| **是否退貨 (目標 y)** | 類別型 (`object`) | **「是」 / 「否」** | **預測目標：訂單是否產生退貨** |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與繁中字型**：確保圖表顯示中文，避免方塊亂碼 (`□□`)。
2. **資料載入與探索 (EDA)**：檢視品類退貨分佈與折扣影響。
3. **特徵關聯性與視覺化**：繪製品類與付款方式之退貨交叉分析長條圖。
4. **特徵工程與 Pipeline 建置**：8:2 切分訓練與測試集，嚴防資料洩漏。
5. **模型選型與 5 折交叉驗證**：比對「邏輯回歸」與「隨機森林分類器」。
6. **測試集大考與逆物流策略**：評估召回率 (Recall) 與混淆矩陣，輸出模型即時預測。

---  
### 【步驟 0】：環境準備與繁體中文字型設定

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入電商退貨風險分析。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入 Pandas、NumPy、Matplotlib、Seaborn 資料科學套件，並設定中文字型庫，確保後續繪製電商退貨圖表時標籤文字不產生方塊亂碼。

---
💡 **電商分析師核心觀念與 AI 提問小抄：**

1. 📌 **中文字型設定原理**
   > 「我是電商營運人員，平常用 Excel 做業績表。請向我解釋為什麼 Python 繪圖預設沒有中文字型？這段程式是如何幫我自動尋找並載入可用的中文字型？」

2. 📌 **負號顯示修正**
   > 「為什麼程式碼中需要加上 `plt.rcParams['axes.unicode_minus'] = False`？如果沒有加這行，在畫負數座標軸時會發生什麼問題？」


---  
### 【步驟 1】：載入資料與電商結構探索 (EDA)

In [ ]:
# [步驟 1 程式碼] 讀取 returns.csv 並檢視資料概況
df = pd.read_csv("returns.csv")
print("資料集維度（列, 欄）：", df.shape)
print("\n--- 資料前 5 筆預覽 ---")
display(df.head())
print("\n--- 各欄位型態與缺失值檢查 ---")
print(df.info())
print("\n--- 退貨標籤分佈比例 ---")
print(df["是否退貨"].value_counts(normalize=True))

#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  讀取 `returns.csv`，檢查總樣本數、欄位資料型態有無異常、是否有漏填缺失值，並統計整體退貨率基準水位。

---
💡 **AI 提問小抄：**

1. 📌 **退貨基準率解讀**
   > 「在我們的 2,000 筆訂單中，退貨率約為 46%。請用電商營運角度告訴我，這個退貨率在不同品類（如服飾 vs 美妝）算正常還是偏高？對毛利有何影響？」

2. 📌 **缺失值檢查重要性**
   > 「如果資料集中有訂單缺少『過往退貨率』或『付款方式』，演算法會直接當機嗎？在電商資料清洗時，一般如何處理新客沒有過往紀錄的情況？」


---  
### 【步驟 2】：特徵關聯分析與品類退貨交叉圖表繪製

In [ ]:
# [步驟 2 程式碼] 繪製各商品類別與付款方式之退貨率長條圖
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
cat_order = df.groupby("商品類別")["是否退貨"].apply(lambda s: (s == "是").mean()).sort_values(ascending=False)
sns.barplot(x=cat_order.index, y=cat_order.values * 100, palette="viridis")
plt.title("各商品類別之退貨率比較 (%)", fontsize=12, pad=10)
plt.xlabel("商品類別")
plt.ylabel("退貨率 (%)")
plt.xticks(rotation=20)

plt.subplot(1, 2, 2)
pay_order = df.groupby("付款方式")["是否退貨"].apply(lambda s: (s == "是").mean()).sort_values(ascending=False)
sns.barplot(x=pay_order.index, y=pay_order.values * 100, palette="magma")
plt.title("不同付款方式之退貨率比較 (%)", fontsize=12, pad=10)
plt.xlabel("付款方式")
plt.ylabel("退貨率 (%)")
plt.xticks(rotation=25)

plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將類別欄位分組計算退貨百分比並繪製直觀長條圖，協助營運者一目了然看出品類與付款方式的逆物流風險差異。

---
💡 **AI 提問小抄：**

1. 📌 **商業因果與關聯剖析**
   > 「從圖表中我們發現『流行服飾』退貨率明顯高於『美妝保養』，而『超商取貨付款』退貨率高於『信用卡線上刷卡』。請用消費者心理學與電商習慣解釋為什麼會有這種現象？」

2. 📌 **圖表優化與修改**
   > 「我想在柱狀圖上方直接標註具體的百分比數值（例如 58.2%），請幫我修改這段 Matplotlib/Seaborn 程式碼加入文字標註標籤。」


---  
### 【步驟 3】：特徵工程與 Pipeline 建構 (防資料洩漏)

In [ ]:
# [步驟 3 程式碼] 切分訓練/測試集，並建立前處理管線
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=["是否退貨"])
y = (df["是否退貨"] == "是").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = ["單價_TWD", "折扣幅度", "過往退貨率", "購買數量"]
cat_cols = ["商品類別", "是否首購顧客", "付款方式", "配送方式"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
])

print(f"✅ 資料分割完成！訓練集：{X_train.shape[0]} 筆，測試集：{X_test.shape[0]} 筆")

#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將數據拆分為 80% 訓練集與 20% 測試集，使用 `StandardScaler` 將金額與折扣標準化，使用 `OneHotEncoder` 將類別轉為虛擬變數，並透過 `ColumnTransformer` 打包，避免測試集答案洩漏。

---
💡 **AI 提問小抄：**

1. 📌 **資料洩漏 (Data Leakage) 防範**
   > 「為什麼特徵標準化（StandardScaler）的 `fit` 必須只能在訓練集上執行，絕對不能在切分前先對整個 DataFrame 做？這跟學生作弊有什麼相似之處？」

2. 📌 **分層抽樣 (Stratified Split)**
   > 「在 `train_test_split` 中加上 `stratify=y` 是什麼意思？如果我們不加，在小樣本或退貨率不均勻時會帶來什麼風險？」


---  
### 【步驟 4】：模型選型與 5 折交叉驗證 (邏輯回歸 vs 隨機森林)

In [ ]:
# [步驟 4 程式碼] 使用 5 折交叉驗證客觀比對兩種演算法
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pipe_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(random_state=42))
])

pipe_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))
])

scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=5, scoring="roc_auc")
scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=5, scoring="roc_auc")

print(f"邏輯回歸 5 折交叉驗證 平均 ROC-AUC: {scores_lr.mean():.4f} (±{scores_lr.std():.4f})")
print(f"隨機森林 5 折交叉驗證 平均 ROC-AUC: {scores_rf.mean():.4f} (±{scores_rf.std():.4f})")

#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  使用 5 折交叉驗證，客觀評估邏輯回歸與隨機森林在訓練集上的泛化能力，避免挑選到運氣好剛好過擬合的模型。

---
💡 **AI 提問小抄：**

1. 📌 **演算法選型思路**
   > 「在電商退貨預測中，為什麼隨機森林的表現通常優於邏輯回歸？隨機森林如何捕捉『服飾類別＋高折扣＋超商取貨』這種多條件的複合非線性特徵？」

2. 📌 **ROC-AUC 指標優勢**
   > 「為什麼在交叉驗證評估時，我們選擇 `scoring='roc_auc'` 而不是單純看正確率（Accuracy）？對於可能存在類別不平衡的退貨問題有何好處？」


---  
### 【步驟 5】：測試集大考與指標解讀 (混淆矩陣, Recall, ROC-AUC)

In [ ]:
# [步驟 5 程式碼] 在全新測試集進行評估
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

pipe_rf.fit(X_train, y_train)
y_pred = pipe_rf.predict(X_test)
y_prob = pipe_rf.predict_proba(X_test)[:, 1]

print("=== 測試集最終分類報告 ===")
print(classification_report(y_test, y_pred, target_names=["未退貨(0)", "退貨(1)"]))
print("測試集 ROC-AUC 分數:", round(roc_auc_score(y_test, y_prob), 4))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["預測未退貨", "預測退貨"], yticklabels=["實際未退貨", "實際退貨"])
plt.title("電商訂單退貨預測混淆矩陣")
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將訓練好的隨機森林模型應用在從未見過的 400 筆測試訂單上，輸出精準度 (Precision)、召回率 (Recall)、F1 分數與混淆矩陣熱力圖。

---
💡 **AI 提問小抄：**

1. 📌 **混淆矩陣白話商業解讀**
   > 「請向電商老闆用大白話解讀這張混淆矩陣中的四個格子：哪些是抓對的退貨、哪些是漏抓的退貨（漏網之魚）、哪些是被誤判的忠實訂單？哪一個數字造成公司的逆物流運費損失最大？」

2. 📌 **決策門檻調整策略**
   > 「目前模型預設以 50% 機率作為退貨切分點。如果我們為了『寧可多發尺寸確認簡訊，也不願多付 120 元逆物流退貨運費』，應該如何把門檻調降到 35%？」


---  
### 【步驟 6】：新訂單即時推論 (Inference) 與模型保存

In [ ]:
# [步驟 6 程式碼] 保存模型管線並模擬新訂單即時預測
import joblib

joblib.dump(pipe_rf, "model.joblib")
print("✅ 模型管線已成功保存至 model.joblib！")

# 模擬一筆即時剛成立的電商訂單
new_order = pd.DataFrame([{
    "商品類別": "流行服飾",
    "單價_TWD": 2480,
    "折扣幅度": 0.35,
    "是否首購顧客": "是",
    "過往退貨率": 0.50,
    "購買數量": 3,
    "付款方式": "超商取貨付款",
    "配送方式": "黑貓宅急便"
}])

loaded_model = joblib.load("model.joblib")
pred_prob = loaded_model.predict_proba(new_order)[0][1]
print(f"\n🔔 新進訂單評估結果：預估退貨機率為 {pred_prob*100:.1f}%")
if pred_prob >= 0.6:
    print("⚠️ 警報：本筆訂單退貨風險極高！建議系統發送 LINE 尺寸確認通知，並於出貨時加強二次質檢。")
else:
    print("🟢 安心訂單：退貨風險低，可安排正常快速出貨。")

#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將訓練好的管線整體保存為 `.joblib` 檔案，隨時可供伺服器或桌面 GUI 應用程式載入，並示範傳入一筆全新訂單即時給予處遇建議。

---
💡 **AI 提問小抄：**

1. 📌 **模型即時部署與 API 串接**
   > 「在真實電商網站（如 SHOPLINE 或 Cyberbiz）中，這段推論程式碼應該埋在結帳後的哪一個時間點觸發？如何串接簡訊與 WMS 倉儲出貨系統？」

2. 📌 **獨立視窗工具應用**
   > 「請告訴我如何配合本範例目錄中的 `app.py`，讓倉庫品檢主管不需要懂 Python 程式碼，也可以直接用滑鼠輸入訂單資訊得到即時建議？」
